# Ballooning mode comparisons

Compare the legacy delta-prime shooting result, the small-solution delta-prime result, and the `ca1` projection result using both flux-surface profiles and a local s-alpha scan.

In [ ]:
using Pkg
Pkg.activate("../..")

using GeneralizedPerturbedEquilibrium
using Plots
using LaTeXStrings
using TOML

eq_config = GeneralizedPerturbedEquilibrium.Equilibrium.EquilibriumConfig(TOML.parsefile("gpec.toml")["Equilibrium"], pwd())
equil = GeneralizedPerturbedEquilibrium.Equilibrium.setup_equilibrium(eq_config)
psi_norm = Vector(equil.profiles.xs)

modes = (:delta_prime_legacy, :delta_prime_small, :ca1)
mode_titles = Dict(
    :delta_prime_legacy => L"\Delta'",
    :delta_prime_small => L"\Delta'",
    :ca1 => L"c_{a1}",
)

real_or_nan(x; atol=1e-10) = x isa Real ? Float64(x) : (x isa Complex && isapprox(imag(x), 0.0; atol=atol) ? Float64(real(x)) : NaN)

function symmetric_clims(z)
    finite_vals = vec(z)[isfinite.(vec(z))]
    maxabs = isempty(finite_vals) ? 1.0 : maximum(abs.(finite_vals))
    maxabs = max(maxabs, eps(Float64))
    return (-maxabs, maxabs)
end


## Flux-surface mode comparison

In [ ]:
psi_indices = collect(eachindex(psi_norm))

mode_values = Dict(mode => fill(NaN, length(psi_indices)) for mode in modes)
di_profile = fill(NaN, length(psi_indices))

for (i, psi_idx) in enumerate(psi_indices)
    coeff = try
        GeneralizedPerturbedEquilibrium.ForceFreeStates.prepare_ballooning_coefficients(psi_idx, equil)
    catch err
        @warn "coefficient preparation failed" psi_idx psi=psi_norm[psi_idx] exception=(err, catch_backtrace())
        nothing
    end
    coeff === nothing && continue

    di_profile[i] = coeff.di

    for mode in modes
        result = try
            GeneralizedPerturbedEquilibrium.ForceFreeStates.integrate_ballooning_ode(
                coeff.ode_coefficient_spline;
                theta_k=0.0,
                mode=mode,
                d0bar=coeff.d0bar,
                n0_spline=coeff.n0_spline,
                n1_spline=coeff.n1_spline,
                theta_grid=coeff.theta_grid,
            )
        catch err
            @warn "mode comparison failed" psi_idx psi=psi_norm[psi_idx] mode exception=(err, catch_backtrace())
            nothing
        end

        if result !== nothing
            mode_values[mode][i] = real_or_nan(result.value)
        end
    end
end

psi_scan = psi_norm[psi_indices]

profile_specs = [
    (:delta_prime_legacy, L"\Delta'", L"\Delta'"),
    (:delta_prime_small, L"\Delta'", L"\Delta'"),
    (:ca1, L"c_{a1}", L"c_{a1}"),
]

profile_plots = map(profile_specs) do (mode, title_text, ylabel_text)
    p = plot(
        psi_scan,
        mode_values[mode];
        xlabel=L"\psi_N",
        ylabel=ylabel_text,
        title=title_text,
        label=String(mode),
        linewidth=2,
        framestyle=:box,
    )
    hline!(p, [0.0]; color=:black, linestyle=:dash, label="")
    p
end

display(plot(profile_plots...; layout=(1, 3), size=(1350, 390)))


## s-alpha mode comparison

In [ ]:
psi_target = 0.67680851
psi_idx_scan = argmin(abs.(psi_norm .- psi_target))

s_scales = collect(range(-5.0, 5.0; length=30))
alpha_scales = collect(range(-5.0, 5.0; length=30))

println("Running s-alpha mode comparison at index $(psi_idx_scan), psi_norm=$(round(psi_norm[psi_idx_scan], digits=6))")

scan_results = Dict{Symbol,Any}()
for mode in modes
    scan_results[mode] = GeneralizedPerturbedEquilibrium.ForceFreeStates.scan_delta_prime_map(
        psi_idx_scan,
        equil;
        theta_k=0.0,
        mode=mode,
        s_scales=s_scales,
        alpha_scales=alpha_scales,
    )
end


In [ ]:
reference = scan_results[:delta_prime_legacy].reference

scan_s_raw = scan_results[:delta_prime_legacy].s_values
scan_alpha_raw = scan_results[:delta_prime_legacy].alpha_values
s_order = sortperm(scan_s_raw)
alpha_order = sortperm(scan_alpha_raw)
scan_s = scan_s_raw[s_order]
scan_alpha = scan_alpha_raw[alpha_order]
di_contour = scan_results[:delta_prime_legacy].di_values[s_order, alpha_order]

salpha_plots = map(modes) do mode
    z = scan_results[mode].delta_prime[s_order, alpha_order]
    p = contourf(
        scan_alpha, scan_s, z;
        c=cgrad([:blue, :white, :red]),
        clims=symmetric_clims(z),
        xlabel=L"\alpha",
        ylabel=L"s",
        title=mode_titles[mode],
        levels=41,
        framestyle=:box,
    )
    contour!(p, scan_alpha, scan_s, z; levels=[0.0], color=:black, linewidth=2, label="", colorbar=false)
    contour!(p, scan_alpha, scan_s, di_contour; levels=[0.0], color=:red, linestyle=:dash, linewidth=2, label=L"D_I=0", colorbar=false)
    scatter!(p, [reference.alpha_ref], [reference.s_ref]; color=:green, marker=:star5, ms=7, label="equilibrium")
    p
end

display(plot(salpha_plots...; layout=(1, 3), size=(1350, 390)))
